# DataScout: Enterprise SQL Intelligence

## The Problem: Data Silos & Privacy
In modern enterprises, valuable data is often locked away in SQL databases. Business stakeholders (Marketing, HR, Sales) need answers, but they lack the SQL skills to query these databases directly. This creates a bottleneck where data analysts are overwhelmed with routine requests.

Existing "Text-to-SQL" solutions often fail in enterprise environments due to:
1. **Security Risks**: Giving an LLM direct access to a database is dangerous.
2. **Privacy Concerns**: Uploading actual data rows to an LLM context window violates privacy compliance (GDPR/SOC2).

## The Solution: DataScout
DataScout is a privacy-first enterprise agent that acts as a secure bridge between business users and their databases. It uses a **"Schema-Handshake"** architecture:

- **Metadata Only**: The Agent receives table schemas (column names/types) but NEVER the actual data rows during the reasoning phase.
- **Secure Execution**: Generated SQL is executed by a sandboxed backend with strict "Read-Only" enforcement.
- **Visual Results**: Data is returned to the frontend for local rendering, keeping raw data out of the LLM's final response text.

## Architecture Overview
The system is built with:
- **Frontend**: Angular 18 (UI, State Management)
- **Backend**: Python FastAPI (Orchestration, Database Connections)
- **Database**: SQLite (Internal Metadata) + Connectors for MySQL, PostgreSQL, MSSQL
- **AI**: Google Gemini (Reasoning & SQL Generation)


## Core Logic: Schema Extraction
The following code demonstrates how DataScout extracts schema information (metadata) without reading the actual data rows. This metadata is what gets sent to the AI agent.

In [ ]:
from sqlalchemy import create_engine, inspect
from typing import List, Dict, Any

def get_schema_from_engine(engine) -> List[Dict[str, Any]]:
    inspector = inspect(engine)
    table_names = inspector.get_table_names()
    schema = []
    for table_name in table_names:
        columns = inspector.get_columns(table_name)
        # columns is a list of dicts with keys: name, type, nullable, etc.
        column_details = []
        for col in columns:
            column_details.append({
                "name": col["name"],
                "type": str(col["type"]),
                "nullable": col.get("nullable", True),
                "primary_key": col.get("primary_key", False)
            })
        schema.append({
            "name": table_name,
            "columns": column_details
        })
    return schema

# Example Usage (Mock)
# engine = create_engine("sqlite:///example.db")
# schema = get_schema_from_engine(engine)
# print(schema)

## Data Models (Pydantic)
To ensure type safety and clear communication between the frontend and backend, we define strict Pydantic models. This ensures that the schema data is always structured correctly before being processed or sent to the AI.

In [ ]:
from pydantic import BaseModel
from typing import List

class ColumnSchema(BaseModel):
    name: str
    type: str
    nullable: bool
    primary_key: bool

class TableSchema(BaseModel):
    name: str
    columns: List[ColumnSchema]

class DatabaseSchemaResponse(BaseModel):
    connection_id: int = None
    tables: List[TableSchema]

## API Layer (FastAPI)
The backend exposes REST endpoints to handle database connections. Here's how we handle a MySQL connection request, extract the schema, and return it to the frontend.

In [ ]:
# Pseudo-code representation of the FastAPI endpoint

# @app.post("/connect/mysql", response_model=DatabaseSchemaResponse)
# def connect_mysql(details: DBConnection):
#     try:
#         # 1. Connect to the user's database
#         schema = get_mysql_schema(details.host, details.port, details.username, details.password, details.database)
#         
#         # 2. Save connection details (encrypted) to internal DB for session persistence
#         conn_id = save_connection_details("mysql", details.dict(), schema)
#         
#         # 3. Return the schema to the frontend
#         return {"connection_id": conn_id, "tables": schema}
#     except Exception as e:
#         raise HTTPException(status_code=400, detail=f"Error connecting to MySQL: {str(e)}")

## Frontend State Management (Angular)
The frontend is responsible for maintaining the state of the schema and handling user interactions (like adding context descriptions). We use an Angular Service with RxJS observables to manage this state reactively.

In [ ]:
// TypeScript Interface for the Frontend
export interface Table {
    name: string;
    context?: string; // User-provided context for the AI
    columns?: Column[];
    isExpanded?: boolean;
}

// Service Logic (Simplified)
// setSchemaData(tables: any[], dbType: string, dbName: string, connectionId: number) {
//     this.currentDbType = dbType;
//     this.currentDbName = dbName;
//     this.currentConnectionId = connectionId;

//     // Transform API response to Table interface
//     this.currentTables = tables.map(t => ({
//         name: t.name,
//         context: t.table_context || '',
//         columns: t.columns.map((c: any) => ({
//             name: c.name,
//             type: c.type,
//             description: '',
//             isLocked: false
//         }))
//     }));
// }

## AI Integration: The Prompt Strategy
The magic happens when we construct the prompt for the AI. We don't send data; we send the schema and the user's question. This allows the AI to generate SQL without ever seeing the sensitive rows.

**Conceptual Prompt Structure:**
```text
You are an expert SQL agent. 
Here is the schema of the database:
[TABLE: users] (columns: id, name, email, signup_date)
[TABLE: orders] (columns: id, user_id, amount, status)

User Question: "How many users signed up last week?"

Task: Generate a SQL query to answer the question. 
Constraint: Read-only queries only. Do not hallucinate columns.
```

## Safety Guardrails
DataScout enforces strict read-only access. Before any query is executed, it passes through a safety check to ensure no destructive commands are present.

In [ ]:
def is_safe_sql(query: str) -> bool:
    forbidden_keywords = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "TRUNCATE", "GRANT", "REVOKE"]
    query_upper = query.upper()
    
    for keyword in forbidden_keywords:
        # Basic check: ensure keyword is not present as a standalone command
        # In production, we use sqlparse for robust tokenization
        if keyword in query_upper:
            return False
            
    return True

# Test
safe_query = "SELECT * FROM users"
unsafe_query = "DROP TABLE users"

print(f"Query: {safe_query} -> Safe? {is_safe_sql(safe_query)}")
print(f"Query: {unsafe_query} -> Safe? {is_safe_sql(unsafe_query)}")

## Project Structure
The full project is structured as follows:

```
DataScout/
├── backend/             # FastAPI Application
│   ├── main.py          # API Endpoints
│   ├── database.py      # Database Logic
│   └── models.py        # Pydantic Models
├── frontend/            # Angular Application
│   ├── src/app/         # Components & Services
│   └── ...
├── scripts/             # Utility Scripts
└── docker-compose.yml   # Container Orchestration
```

## Conclusion
DataScout demonstrates how modern AI agents can be safely integrated into enterprise workflows by decoupling the reasoning layer (LLM) from the data layer (Database), ensuring privacy, security, and accuracy.